```markdown
# Treinamento YOLO11m para Detecção de Tráfego com Google Drive
**Google Colab - GPU Gratuita**

Este notebook está configurado para utilizar o seu dataset personalizado do Google Drive (`traffic_dataset.zip`).

## Como usar:
1. Ative a GPU: **Runtime → Change runtime type → GPU → Save**
2. Execute cada célula em ordem
3. Baixe o modelo treinado ao final

**Tempo estimado:** 30-60 minutos para 50 épocas com GPU.
```

In [6]:
# ============================================================================
# PASSO 1: Instalar dependências
# ============================================================================
!pip install -q ultralytics opencv-python numpy
print("✅ Dependências instaladas")

✅ Dependências instaladas


In [7]:
# ============================================================================
# PASSO 2: Carregar e extrair o dataset do Google Drive
# ============================================================================
import os
import shutil
from google.colab import drive

print('🔗 Conectando ao Google Drive...')
drive.mount('/content/drive')

drive_zip_path = '/content/drive/MyDrive/traffic_dataset.zip'
local_zip_path = '/content/dataset.zip'
extract_path = '/content/dataset'

if os.path.exists(drive_zip_path):
    # Limpar extrações antigas para evitar mistura de dados
    if os.path.exists(extract_path):
        print('🧹 Removendo dataset extraído antigo...')
        shutil.rmtree(extract_path)
    if os.path.exists(local_zip_path):
        os.remove(local_zip_path)

    print(f'📥 Copiando {drive_zip_path} para o ambiente local...')
    shutil.copy(drive_zip_path, local_zip_path)

    print('📦 Extraindo dataset...')
    !unzip -q {local_zip_path} -d {extract_path}
    print('✅ Dataset extraído com sucesso na pasta /content/dataset!')
else:
    print(f'❌ Erro: O arquivo {drive_zip_path} não foi encontrado no seu Google Drive.')
    print("Certifique-se de que o nome do arquivo seja exatamente 'traffic_dataset.zip'.")

🔗 Conectando ao Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📥 Copiando /content/drive/MyDrive/traffic_dataset.zip para o ambiente local...
📦 Extraindo dataset...
✅ Dataset extraído com sucesso na pasta /content/dataset!


In [13]:
# ============================================================================
# EXTRA: Limpar pastas indesejadas no diretório raiz
# ============================================================================
import shutil
import os

pastas_para_remover = ['/content/labels', '/content/images']

print("🧹 Iniciando limpeza de pastas indesejadas fora do dataset...")
for pasta in pastas_para_remover:
    if os.path.exists(pasta):
        try:
            shutil.rmtree(pasta)
            print(f"✅ Pasta removida com sucesso: {pasta}")
        except Exception as e:
            print(f"❌ Erro ao remover a pasta {pasta}: {e}")
    else:
        print(f"ℹ️ A pasta {pasta} não existe ou já foi removida.")

🧹 Iniciando limpeza de pastas indesejadas fora do dataset...
✅ Pasta removida com sucesso: /content/labels
✅ Pasta removida com sucesso: /content/images


In [10]:
# ============================================================================
# PASSO 3: Criar arquivo data.yaml
# ============================================================================
import os

# Caminhos corrigidos apontando para a pasta correta extraída do seu zip
yaml_content = '''train: /content/dataset/traffic_dataset/images/train
val: /content/dataset/traffic_dataset/images/val
nc: 6
names:
  0: person
  1: car
  2: motorcycle
  3: bus
  4: truck
  5: articulated_truck'''

with open('data.yaml', 'w') as f:
    f.write(yaml_content)

print('✅ Arquivo data.yaml configurado e criado com os caminhos corretos:')
print(yaml_content)

✅ Arquivo data.yaml configurado e criado com os caminhos corretos:
train: /content/dataset/traffic_dataset/images/train
val: /content/dataset/traffic_dataset/images/val
nc: 6
names:
  0: person
  1: car
  2: motorcycle
  3: bus
  4: truck
  5: articulated_truck


In [11]:
# ============================================================================
# PASSO 4: Treinar o modelo
# ============================================================================
import torch
from ultralytics import YOLO

device = '0' if torch.cuda.is_available() else 'cpu'
if device == 'cpu':
    print('⚠️ AVISO: GPU não detectada. Treinando em CPU (será extremamente lento).')
else:
    print(f'✅ GPU detectada: {torch.cuda.get_device_name(0)}')

model = YOLO('yolo11m.pt')

results = model.train(
    data='data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=device,
    project='runs/train',
    name='traffic_analysis',
    exist_ok=True,
    patience=20,
    save=True,
    verbose=True
)

print('\n✅ Treinamento concluído!')
print(f"📁 Modelo salvo em: runs/train/traffic_analysis/weights/best.pt")

✅ GPU detectada: Tesla T4
Ultralytics 8.4.220 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=traffic_analysis, nbs=64, nms=None, opset=None, optim

In [14]:
# ============================================================================
# PASSO 5: Validar o modelo treinado
# ============================================================================
from ultralytics import YOLO

model = YOLO('runs/detect/runs/train/traffic_analysis/weights/best.pt')

metrics = model.val(
    data='data.yaml',
    imgsz=640,
    batch=16,
    device=device
)

print('\n✅ Validação concluída!')
print(f"   mAP50: {metrics.box.map50:.4f}")
print(f"   mAP50-95: {metrics.box.map:.4f}")
print(f"   Precisão: {metrics.box.mp:.4f}")
print(f"   Recall: {metrics.box.mr:.4f}")

Ultralytics 8.4.220 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11m summary (fused): 125 layers, 20,034,658 parameters, 0 gradients, 67.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1194.4±368.3 MB/s, size: 40.7 KB)
val: Scanning /content/dataset/traffic_dataset/labels/val.cache... 500 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 500/500 37.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 3.0it/s 10.6s
                   all        500       1984      0.999      0.993      0.992      0.983
                   car        298        431      0.999          1          1      0.995
            motorcycle        286        401          1       0.97       0.97      0.934
                   bus        283        392      0.997      0.999          1      0.997
                 truck        270        361      0.998      0.997       0.99      0.989
     articulated_truck      

In [16]:
# ============================================================================
# PASSO 6: Baixar o modelo treinado
# ============================================================================
from google.colab import files
import os

model_path = 'runs/detect/runs/train/traffic_analysis/weights/best.pt'

if os.path.exists(model_path):
    print(f"📥 Baixando modelo: {model_path}")
    files.download(model_path)
    print('✅ Download iniciado!')
else:
    print('❌ Modelo não encontrado. Verifique se o treinamento foi concluído.')

📥 Baixando modelo: runs/detect/runs/train/traffic_analysis/weights/best.pt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download iniciado!


```markdown
# 📋 Como usar o modelo treinado no seu projeto

1. Baixe o arquivo `best.pt` que foi gerado acima
2. Substitua o arquivo `models/yolo11m.pt` no seu projeto pelo novo modelo
3. Execute normalmente:
   ```bat
   run.bat --mode vehicles --source assets/seu_video.mp4 --no-show
   ```

---
## 📚 Dicas para melhorar o resultado:
- Use pelo menos 100-200 imagens anotadas para treino
- Variedade de ângulos, iluminação e clima
- Anote apenas as classes que você precisa (car, truck, bus, motorcycle, person)
- Use `--conf 0.25` para detecção padrão ou `--conf 0.40` para detecção mais conservadora

---
🏁 **Fim do notebook**
```